In [3]:
%load_ext autoreload
%autoreload 2

from source.dataset import GraphDataset, CombinedDataset, GraphDataModule
from source.model import GravNetModel


In [4]:
import glob
import tqdm
import torch
from torch_geometric.data import DataLoader
import torch.nn as nn 
import torch.nn.functional as F
# from source.train import GravNetLightning
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import pytorch_lightning as pl
from pytorch_lightning import Trainer
import os
from source.preprocess import preprocess_data

In [5]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10000/*.root")
# preprocess_data(root_files, "data/pt/10000/")

In [6]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10001/*.root")
# preprocess_data(root_files, "data/pt/10001/")

In [7]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10002/*.root")
# preprocess_data(root_files, "data/pt/10002/")

In [8]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10003/*.root")
# preprocess_data(root_files, "data/pt/10003/")

In [ ]:
class GravNetLightning(pl.LightningModule):
    def __init__(
        self,
        lr=1e-3,
        lambda_reg=0.5,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.model = GravNetModel()

        self.cls_loss = nn.CrossEntropyLoss()
        self.reg_loss = nn.HuberLoss()

    def forward(self, data):
        return self.model(data)

    def training_step(self, batch, batch_idx):
        
        if batch_idx % 50 == 0:
            print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")


        class_logits, energy_pred = self(batch)

        cls_loss = self.cls_loss(class_logits, batch.y_class)
        reg_loss = self.reg_loss(energy_pred, batch.y_energy)

        loss = cls_loss + self.hparams.lambda_reg * reg_loss

        preds = torch.argmax(class_logits, dim=1)
        acc = (preds == batch.y_class).float().mean()

        self.log("train_loss", loss.detach(), prog_bar=True, batch_size=batch.num_graphs)
        self.log("train_cls_loss", cls_loss.detach(), batch_size=batch.num_graphs)
        self.log("train_reg_loss", reg_loss.detach(), batch_size=batch.num_graphs)
        self.log("train_acc", acc.detach(), prog_bar=True, batch_size=batch.num_graphs)


        return loss

    def validation_step(self, batch, batch_idx):
        class_logits, energy_pred = self(batch)

        cls_loss = self.cls_loss(class_logits, batch.y_class)
        reg_loss = self.reg_loss(energy_pred, batch.y_energy)

        loss = cls_loss + self.hparams.lambda_reg * reg_loss

        preds = torch.argmax(class_logits, dim=1)
        acc = (preds == batch.y_class).float().mean()

        self.log("val_loss", loss.detach(), prog_bar=True, batch_size=batch.num_graphs)
        self.log("val_acc", acc.detach(), prog_bar=True, batch_size=batch.num_graphs)


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=50,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler,
        }




In [ ]:
model = GravNetLightning()

# logger = TensorBoardLogger()
        # save_dir=cfg.logging.log_dir,
        # save_dir=hydra.core.hydra_config.HydraConfig.get().runtime.output_dir,    )

pt_files = glob.glob("data/pt/*/*.pt")
assert len(pt_files) > 0, "No .pt files found" 

datamodule = GraphDataModule(
    pt_files=pt_files,
    batch_size=1,
    num_workers=1,
)


checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min",
)  

early_stop_cb = EarlyStopping(
monitor="val_loss",
min_delta=0,
patience=12,
mode="min",
verbose=True,
)


trainer = Trainer(
    max_epochs=1000,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb],
    log_every_n_steps=50,
    accumulate_grad_batches=1
)

trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type             | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | model    | GravNetModel     | 54.1 K | train | 0    
1 | cls_loss | CrossEntropyLoss | 0      | train | 0    
2 | reg_loss | HuberLoss        | 0      | train | 0    
--------------------------------------------------------------
54.1 K    Trainable params
0         Non-trainable params
54.1 K    Total params
0.217     Total estimated model params size (MB)
53        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

2.485417984 GB
2.494664704 GB
2.49499648 GB
2.494722048 GB
2.494769664 GB
2.49464576 GB
2.494723584 GB
2.494590976 GB
2.494987264 GB
2.4956544 GB
2.494600704 GB
2.494813184 GB
2.495102464 GB
2.494685184 GB
2.494802944 GB
2.494641664 GB
2.494654976 GB
2.494974464 GB
2.494586368 GB
2.494792704 GB
2.494621184 GB
2.4946816 GB
2.494813184 GB
2.49474816 GB
2.49467392 GB
2.495389184 GB
2.497055744 GB
2.497797632 GB
2.494690816 GB
2.494639104 GB
2.494769664 GB
2.49532416 GB
2.49458432 GB
2.49488128 GB
2.49684992 GB
2.495056896 GB
2.494772224 GB
2.49485568 GB
2.496180736 GB
2.495281664 GB
2.494928384 GB
2.49499136 GB
2.494821376 GB
2.49488128 GB
2.49475328 GB
2.4958208 GB
2.494662144 GB
2.49554944 GB
2.495147008 GB
2.494682624 GB
2.49486848 GB
2.49527808 GB
2.494687744 GB
2.4946304 GB
2.494669824 GB
2.49466368 GB
2.495187456 GB
2.494898688 GB
2.494600704 GB
2.495115264 GB
2.49458432 GB
2.49464832 GB
2.49543424 GB
2.4951168 GB
2.4946432 GB
2.494943744 GB
2.494796288 GB
2.494629888 GB
2.494646784


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

: 